In [3]:
import os
import pandas as pd
import sqlite3
from tqdm import tqdm

root_folder = 'C:/Users/20232075/Desktop/London Police Data'
db_path = 'crime_data.db'
batch_size = 2000

required_columns = [
    'Crime ID', 'Month', 'Reported by', 'Falls within',
    'Longitude', 'Latitude', 'Location',
    'LSOA code', 'Crime type', 'Last outcome category', 'Context'
]

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("PRAGMA journal_mode=WAL;")

cursor.execute('''
    CREATE TABLE IF NOT EXISTS crime (
        crimeID TEXT PRIMARY KEY,
        Month TEXT,
        Longitude REAL,
        Latitude REAL,
        LSOA_code TEXT,
        Type TEXT,
        Outcome TEXT
    )
''')
conn.commit()

insert_query = '''
    INSERT OR IGNORE INTO crime (
        crimeID, Month,Longitude, Latitude,
        LSOA_code, Type, Outcome
    ) VALUES (?, ?, ?, ?, ?, ?, ?);
'''

all_files = []
for subdir, dirs, files in os.walk(root_folder):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(subdir, file))

batch = []
file_count = 0
inserted_rows = 0

for file_path in tqdm(all_files, desc="Processing files", unit="file"):
    name_without_ext = file_path[:-4]
    if name_without_ext.lower().endswith('-street'):
        df = pd.read_csv(file_path)
        for col in required_columns:
            if col not in df.columns:
                df[col] = None

        df = df.rename(columns={
            'Crime ID': 'crimeID',
            'Month': 'Month',
            'Longitude': 'Longitude',
            'Latitude': 'Latitude',
            'LSOA code': 'LSOA_code',
            'Crime type': 'Type',
            'Last outcome category': 'Outcome',
        })

        df = df[['crimeID', 'Month', 'Longitude', 'Latitude',
                  'LSOA_code', 'Type', 'Outcome',]]

        records = list(df.itertuples(index=False, name=None))

        for record in tqdm(records, desc=f"Inserting {os.path.basename(file_path)}", leave=False):
            batch.append(record)
            if len(batch) >= batch_size:
                cursor.executemany(insert_query, batch)
                conn.commit()
                inserted_rows += len(batch)
                batch = []

        file_count += 1

if batch:
    cursor.executemany(insert_query, batch)
    conn.commit()
    inserted_rows += len(batch)

conn.close()
print(f"\nInserted {inserted_rows} rows from {file_count} files.")


Processing files: 100%|██████████| 72/72 [03:21<00:00,  2.79s/file]


Inserted 3386817 rows from 36 files.


In [4]:
# Reconnect to the DB
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

for file_path in tqdm(all_files, desc="Processing outcome files", unit="file"):
    name_without_ext = file_path[:-4]
    if name_without_ext.lower().endswith('-outcomes'):
        df = pd.read_csv(file_path)
        df = df[['Crime ID', 'Outcome type']].dropna(subset=['Crime ID'])

        update_records = list(df.itertuples(index=False, name=None))

        for crime_id, outcome in tqdm(update_records, desc=f"Updating {os.path.basename(file_path)}", leave=False):
            cursor.execute(
                "UPDATE crime SET Outcome = ? WHERE crimeID = ?;",
                (outcome, crime_id)
            )

conn.commit()
conn.close()
print("Outcome fields updated where applicable.")


Processing outcome files: 100%|██████████| 72/72 [01:24<00:00,  1.18s/file]


Outcome fields updated where applicable.


In [11]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) from crime"           )
result = cursor.fetchall()
print(result)

[(3309375,)]
